<a href="https://colab.research.google.com/github/elenasabbioni/SMARTbiomed_summerschool_2026/blob/main/session4/practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikbaya/smartbiomed_practicals_2026/blob/main/session4/practical.ipynb)

# Session 4: Fine-mapping

**Timing**: \~60 minutes (Parts 1–3 \~35 min; challenges for fast finishers).

**Dataset**: two simulated GWAS **loci** (\~400 variants each, N=5,000) with realistic LD.
- Locus A has **one** causal variant hidden in a cluster of near-perfectly-correlated SNPs.
- Locus B has **two** causal variants.

The goal of fine-mapping: go from "dozens of significant variants in a peak" to a short list of
**credible** causal candidates. Everything is built from regression + a one-line Bayes factor.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — run once. Downloads the data (on Colab) and defines run_gwas + helpers.
# ─────────────────────────────────────────────────────────────────────────────
import os, numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 80, 'font.size': 11})

# Locate the data directory, downloading from GitHub if needed (e.g. on Colab).
import os, urllib.request
_NEED = ['finemap_data.npz']
_LFS  = {'gwas_data.npz', 'sumstats_real.npz', 'pca_data.npz', 'finemap_data.npz'}   # Git LFS
def _has_all(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in _NEED)
DATA_DIR = next((d for d in ('../data', 'data') if _has_all(d)), None)
if DATA_DIR is None:
    DATA_DIR = 'data'; os.makedirs(DATA_DIR, exist_ok=True)
    for _f in _NEED:
        _dest = os.path.join(DATA_DIR, _f)
        if os.path.exists(_dest):
            continue
        _base = ('https://media.githubusercontent.com/media' if _f in _LFS
                 else 'https://raw.githubusercontent.com')
        _url = f'{_base}/nikbaya/smartbiomed_practicals_2026/main/data/{_f}'
        print(f'Downloading {_f} from GitHub ...')
        urllib.request.urlretrieve(_url, _dest)

d = np.load(os.path.join(DATA_DIR, 'finemap_data.npz'), allow_pickle=True)
GA = d['GA'].astype(np.float32); yA = d['yA']; posA = d['posA']; causalA = int(d['causalA'][0])
GB = d['GB'].astype(np.float32); yB = d['yB']; posB = d['posB']; causalB = d['causalB']
print(f"Locus A: {GA.shape[1]} variants, N={GA.shape[0]:,}, 1 causal")
print(f"Locus B: {GB.shape[1]} variants, N={GB.shape[0]:,}, 2 causal")

def run_gwas(y, G, covars=None, chunk=5_000):
    """
    Vectorised OLS GWAS: regress phenotype y on each column of G.
    Processes variants in batches to keep peak memory usage low.

    Parameters
    ----------
    y      : (N,)    phenotype (will be mean-centred internally)
    G      : (N, M)  genotype matrix (0/1/2), NaN = missing (treated as mean)
    covars : (N, k)  covariate matrix (optional); age and sex recommended
    chunk  : int     variants per batch (default 5,000)

    Returns
    -------
    betas  : (M,)  per-variant OLS effect size estimate
    ses    : (M,)  standard error of beta
    pvals  : (M,)  two-sided p-value
    """
    N, M = G.shape
    if covars is None:
        C = np.ones((N, 1))
    else:
        C = np.column_stack([np.ones(N), covars])

    # Residualise y on covariates once (cheap)
    Q, _  = np.linalg.qr(C, mode='reduced')
    y_r   = y - Q @ (Q.T @ y)
    ss_y  = float(np.dot(y_r, y_r))
    n_df  = N - C.shape[1] - 1

    betas = np.empty(M); ses = np.empty(M)

    for s in range(0, M, chunk):
        e    = min(s + chunk, M)
        G_c  = G[:, s:e].astype(float)
        mu   = np.nanmean(G_c, axis=0)
        ri, ci = np.where(np.isnan(G_c)); G_c[ri, ci] = mu[ci]   # mean-impute
        G_r  = G_c - Q @ (Q.T @ G_c)                              # residualise
        ss_G = (G_r**2).sum(0)
        b    = G_r.T @ y_r / ss_G
        betas[s:e] = b
        rss  = ss_y - b**2 * ss_G
        ses[s:e]   = np.sqrt(np.clip(rss, 0, None) / n_df / ss_G)

    t_stats = betas / (ses + 1e-300)
    pvals   = 2 * stats.t.sf(np.abs(t_stats), df=n_df)
    return betas, ses, pvals

def r2_with(G, j):
    """r² between variant j and every variant in the locus."""
    g = G[:, j]
    return np.array([np.corrcoef(g, G[:, k])[0, 1]**2 for k in range(G.shape[1])])

print("Setup ready: run_gwas(), r2_with().")


Locus A: 400 variants, N=5,000, 1 causal
Locus B: 400 variants, N=5,000, 2 causal
Setup ready: run_gwas(), r2_with().


## Part 1: Why a single GWAS peak isn't enough

Inside an associated locus, **many** variants are genome-wide significant — not because they are
all causal, but because they are in **LD** with the causal one. A LocusZoom-style plot (coloured
by $r^2$ with the lead SNP) shows the tell-tale triangular peak. The marginal **lead** SNP is often
just a *tag*, not the causal variant.


In [ ]:
# Exercise 1.1: regional GWAS + LocusZoom of Locus A.
# YOUR CODE HERE: run the GWAS of phenotype yA on the locus genotypes GA.
betas, std_errors, pvalues = run_gwas(???, ???)

lead_variant = int(np.argmin(pvalues))          # most significant ("lead") variant
r2_to_lead   = r2_with(GA, lead_variant)         # LD (r²) of every variant with the lead

print(f"Genome-wide significant variants: {(pvalues < 5e-8).sum()} of {len(pvalues)}")
print(f"Marginal lead = variant {lead_variant} (pos {posA[lead_variant]} kb);  "
      f"true causal = variant {causalA}")

fig, ax = plt.subplots(figsize=(11, 4))
sc = ax.scatter(posA, -np.log10(pvalues + 1e-300), c=r2_to_lead, cmap='RdYlBu_r',
                vmin=0, vmax=1, s=18)
ax.scatter(posA[causalA], -np.log10(pvalues[causalA] + 1e-300), marker='*', s=320,
           color='lime', edgecolor='black', zorder=5, label='true causal')
plt.colorbar(sc, ax=ax, label='$r^2$ with lead'); ax.axhline(-np.log10(5e-8), color='red', ls='--')
ax.set_xlabel('Position (kb)'); ax.set_ylabel(r'$-\log_{10}p$')
ax.set_title('Locus A — many significant variants in one LD peak'); ax.legend()
plt.tight_layout(); plt.show()
print("Q: How many variants would you have to follow up if you took every significant one?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

betas, std_errors, pvalues = run_gwas(yA, GA)

lead_variant = int(np.argmin(pvalues))
r2_to_lead   = r2_with(GA, lead_variant)

print(f"Genome-wide significant variants: {(pvalues < 5e-8).sum()} of {len(pvalues)}")
print(f"Marginal lead = variant {lead_variant} (pos {posA[lead_variant]} kb);  "
      f"true causal = variant {causalA}")

fig, ax = plt.subplots(figsize=(11, 4))
sc = ax.scatter(posA, -np.log10(pvalues + 1e-300), c=r2_to_lead, cmap='RdYlBu_r',
                vmin=0, vmax=1, s=18)
ax.scatter(posA[causalA], -np.log10(pvalues[causalA] + 1e-300), marker='*', s=320,
           color='lime', edgecolor='black', zorder=5, label='true causal')
plt.colorbar(sc, ax=ax, label='$r^2$ with lead'); ax.axhline(-np.log10(5e-8), color='red', ls='--')
ax.set_xlabel('Position (kb)'); ax.set_ylabel(r'$-\log_{10}p$')
ax.set_title('Locus A — many significant variants in one LD peak'); ax.legend()
plt.tight_layout(); plt.show()


<details>
<summary>👉 <b>Show answer</b></summary>

All of them — every variant in the LD block around the causal SNP crosses genome-wide significance, so naively you'd have to follow up the **whole peak** (often dozens to hundreds of variants). That is the problem fine-mapping solves: LD makes many non-causal variants look significant.

</details>

## Part 2: "Poor man's" fine-mapping — conditional analysis

A simple way to count **independent** signals: take the lead variant, add it as a **covariate**,
and re-run the GWAS. If a signal remains, there's a second independent association; repeat until
nothing is significant. (This isn't a credible set, but it builds the right intuition.)

We'll **re-draw the LocusZoom after every round**, so you can watch the peak collapse once its
causal variant is conditioned out.


In [ ]:
# Exercise 2.1: iterative conditional analysis on Locus A, with a LocusZoom after each round.

# A small helper that draws one LocusZoom-style plot (−log10 p, coloured by r² with the round's lead).
def plot_locuszoom(pvalues, lead_variant, title):
    r2_to_lead = r2_with(GA, lead_variant)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    sc = ax.scatter(posA, -np.log10(pvalues + 1e-300), c=r2_to_lead,
                    cmap='RdYlBu_r', vmin=0, vmax=1, s=16)
    ax.scatter(posA[causalA], -np.log10(pvalues[causalA] + 1e-300), marker='*', s=280,
               color='lime', edgecolor='black', zorder=5, label='true causal')
    ax.axvline(posA[lead_variant], color='grey', ls=':', label=f'round lead (variant {lead_variant})')
    ax.axhline(-np.log10(5e-8), color='red', ls='--')
    plt.colorbar(sc, ax=ax, label='r² with round lead')
    ax.set_xlabel('Position (kb)'); ax.set_ylabel(r'$-\log_{10}p$')
    ax.set_title(title); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

significance_threshold = 5e-8
conditioned_on = []                         # variants we have added as covariates so far

# Round 0: the ordinary GWAS, with nothing conditioned out yet.
_, _, pvalues = run_gwas(yA, GA)
round_number = 0

while round_number < 6:                     # safety cap on the number of rounds
    lead_variant = int(np.argmin(pvalues))
    conditioning_text = conditioned_on if conditioned_on else 'nothing'
    plot_locuszoom(pvalues, lead_variant,
                   f'Round {round_number}: conditioning on {conditioning_text}  '
                   f'(lead variant {lead_variant}, p = {pvalues[lead_variant]:.1e})')

    # Stop once the strongest remaining signal is no longer genome-wide significant.
    if pvalues[lead_variant] >= significance_threshold:
        print("No genome-wide-significant signal remains — stop.")
        break

    # Otherwise, record this lead and condition on ALL signals found so far.
    conditioned_on.append(lead_variant)
    signal_genotypes = GA[:, conditioned_on]    # genotype columns of every signal so far
    # YOUR CODE HERE: re-run the GWAS, passing the signal genotypes as covariates.
    _, _, pvalues = run_gwas(yA, GA, covars=???)
    round_number += 1

print(f"Independent signals found: {len(conditioned_on)}  → variants {conditioned_on}")
print("(Locus A has 1 causal variant — do you recover a single independent signal?)")
print("Q: After conditioning on the lead, why do all the other peak variants drop out?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

def plot_locuszoom(pvalues, lead_variant, title):
    r2_to_lead = r2_with(GA, lead_variant)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    sc = ax.scatter(posA, -np.log10(pvalues + 1e-300), c=r2_to_lead,
                    cmap='RdYlBu_r', vmin=0, vmax=1, s=16)
    ax.scatter(posA[causalA], -np.log10(pvalues[causalA] + 1e-300), marker='*', s=280,
               color='lime', edgecolor='black', zorder=5, label='true causal')
    ax.axvline(posA[lead_variant], color='grey', ls=':', label=f'round lead (variant {lead_variant})')
    ax.axhline(-np.log10(5e-8), color='red', ls='--')
    plt.colorbar(sc, ax=ax, label='r² with round lead')
    ax.set_xlabel('Position (kb)'); ax.set_ylabel(r'$-\log_{10}p$')
    ax.set_title(title); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

significance_threshold = 5e-8
conditioned_on = []
_, _, pvalues = run_gwas(yA, GA)
round_number = 0
while round_number < 6:
    lead_variant = int(np.argmin(pvalues))
    conditioning_text = conditioned_on if conditioned_on else 'nothing'
    plot_locuszoom(pvalues, lead_variant,
                   f'Round {round_number}: conditioning on {conditioning_text}  '
                   f'(lead variant {lead_variant}, p = {pvalues[lead_variant]:.1e})')
    if pvalues[lead_variant] >= significance_threshold:
        print("No genome-wide-significant signal remains — stop.")
        break
    conditioned_on.append(lead_variant)
    signal_genotypes = GA[:, conditioned_on]
    _, _, pvalues = run_gwas(yA, GA, covars=signal_genotypes)
    round_number += 1
print(f"Independent signals found: {len(conditioned_on)}  → variants {conditioned_on}")
# Conditioning on the lead absorbs the shared LD signal, so its tag SNPs are no longer significant.


<details>
<summary>👉 <b>Show answer</b></summary>

Because the other peak variants were only significant through **LD with the lead** variant — once you condition on (regress out) the lead, the signal they were borrowing disappears and their p-values collapse. This is evidence the locus holds a **single** causal signal that the lead variant tags.

</details>

## Part 3: Posterior probabilities and the credible set

Bayesian fine-mapping assigns each variant a **posterior inclusion probability (PIP)** — the
probability it is the causal one. Assuming a single causal variant in the locus, Wakefield's
**approximate Bayes factor (ABF)** for variant $j$ uses only its $z = \hat\beta/\text{se}$:

$$\log\text{ABF}_j = \tfrac{1}{2}\log(1-r) + \tfrac{1}{2} z_j^2\, r, \qquad r = \frac{W}{V_j + W}$$

where $V_j=\text{se}_j^2$ and $W$ is the prior variance of the effect. Normalising the ABFs gives
PIPs; the **95% credible set** is the smallest set of variants whose PIPs sum to ≥ 0.95.


In [ ]:
# Exercise 3.1: PIP and the 95% credible set for Locus A.
betas, std_errors, pvalues = run_gwas(yA, GA)
z = betas / std_errors

# Wakefield approximate Bayes factor, one variant at a time.
prior_variance    = 0.04                     # W: prior variance of the true effect size
sampling_variance = std_errors**2            # V: squared standard error of each estimate
r = prior_variance / (sampling_variance + prior_variance)

# YOUR CODE HERE: the log approximate Bayes factor = 0.5*log(1 - r) + 0.5 * z^2 * r
log_abf = ???
abf     = np.exp(log_abf - log_abf.max())    # exponentiate (subtract max for numerical stability)
# YOUR CODE HERE: posterior inclusion probabilities = abf normalised to sum to 1
pip     = ???

# 95% credible set: the smallest set of top-PIP variants whose PIPs sum to >= 0.95.
order          = np.argsort(pip)[::-1]                 # variants from highest to lowest PIP
cumulative_pip = np.cumsum(pip[order])
credible_set   = order[:np.searchsorted(cumulative_pip, 0.95) + 1]

print(f"95% credible set: {sorted(credible_set.tolist())}  ({len(credible_set)} variants)")
print(f"Contains the true causal ({causalA})? {causalA in credible_set.tolist()}")
print(f"PIP of the true causal: {pip[causalA]:.2f}   (max PIP in locus: {pip.max():.2f})")
print("Q: Fine-mapping cut the significant peak down to how few variants?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

betas, std_errors, pvalues = run_gwas(yA, GA)
z = betas / std_errors

prior_variance    = 0.04
sampling_variance = std_errors**2
r = prior_variance / (sampling_variance + prior_variance)

log_abf = 0.5*np.log(1 - r) + 0.5*z**2*r
abf     = np.exp(log_abf - log_abf.max())
pip     = abf / abf.sum()

order          = np.argsort(pip)[::-1]
cumulative_pip = np.cumsum(pip[order])
credible_set   = order[:np.searchsorted(cumulative_pip, 0.95) + 1]

print(f"95% credible set: {sorted(credible_set.tolist())}  ({len(credible_set)} variants)")
print(f"Contains the true causal ({causalA})? {causalA in credible_set.tolist()}")
print(f"PIP of the true causal: {pip[causalA]:.2f}   (max PIP in locus: {pip.max():.2f})")


<details>
<summary>👉 <b>Show answer</b></summary>

Fine-mapping cuts the peak down to a **handful** of variants — the 95% credible set, often just a few SNPs that together capture 95% of the posterior probability of being causal. It turns 'hundreds of significant variants' into a short, follow-up-able list.

</details>

---

## Challenge Questions


### Challenge 1: A PIP LocusZoom — \~10 min

Re-draw the locus with each variant's height = PIP (instead of −log10 p). The credible set should
stand out as the handful of high-PIP variants. Mark the true causal.


In [ ]:
# Challenge: PIP LocusZoom for Locus A  (reuse pip, credible_set from Part 3).
is_in_credible_set = np.zeros(len(pip), bool)
is_in_credible_set[credible_set] = True

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(posA[~is_in_credible_set], pip[~is_in_credible_set], s=15, color='lightgrey',
           label='not in credible set')
ax.scatter(posA[is_in_credible_set], pip[is_in_credible_set], s=40, color='#e15759', zorder=4,
           label='95% credible set')
ax.scatter(posA[causalA], pip[causalA], marker='*', s=320, color='lime',
           edgecolor='black', zorder=5, label='true causal')
ax.set_xlabel('Position (kb)'); ax.set_ylabel('PIP'); ax.set_title('Fine-mapping: PIP by position')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Sum of PIP over the credible set: {pip[credible_set].sum():.2f}")
print("Q: The whole significant peak collapses to a few high-PIP variants — why those?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

is_in_credible_set = np.zeros(len(pip), bool)
is_in_credible_set[credible_set] = True

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(posA[~is_in_credible_set], pip[~is_in_credible_set], s=15, color='lightgrey',
           label='not in credible set')
ax.scatter(posA[is_in_credible_set], pip[is_in_credible_set], s=40, color='#e15759', zorder=4,
           label='95% credible set')
ax.scatter(posA[causalA], pip[causalA], marker='*', s=320, color='lime',
           edgecolor='black', zorder=5, label='true causal')
ax.set_xlabel('Position (kb)'); ax.set_ylabel('PIP'); ax.set_title('Fine-mapping: PIP by position')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Sum of PIP over the credible set: {pip[credible_set].sum():.2f}")


<details>
<summary>👉 <b>Show answer</b></summary>

Those few variants have the **highest posterior inclusion probability (PIP)** because they best explain the observed association pattern given the LD structure — they sit closest to the causal variant and aren't fully redundant with a stronger neighbour. The rest are explained away as LD partners.

</details>

### Challenge 2: Resolution depends on power — \~10 min

Fine-mapping resolution improves with sample size. Recompute the credible set on random subsets of
the individuals (N/2, N/4) and watch it **grow** as power falls.


In [ ]:
# Challenge: credible-set size vs sample size.
def compute_credible_set(phenotype, genotypes, prior_variance=0.04):
    """Return the 95% credible set (array of variant indices) for a locus."""
    betas, std_errors, _ = run_gwas(phenotype, genotypes)
    z = betas / std_errors
    sampling_variance = std_errors**2
    r = prior_variance / (sampling_variance + prior_variance)
    log_abf = 0.5*np.log(1 - r) + 0.5*z**2*r
    pip = np.exp(log_abf - log_abf.max()); pip /= pip.sum()
    order = np.argsort(pip)[::-1]
    cumulative_pip = np.cumsum(pip[order])
    return order[:np.searchsorted(cumulative_pip, 0.95) + 1]

rng = np.random.default_rng(0)
for fraction in [1.0, 0.5, 0.25]:
    n = int(fraction * len(yA))
    subsample = rng.choice(len(yA), n, replace=False)
    # YOUR CODE HERE: compute the credible set on this subsample of individuals.
    cs = compute_credible_set(???, ???)
    print(f"N={n:5d}: credible-set size = {len(cs):3d}, contains causal = {causalA in cs.tolist()}")
print("Q: Why does a smaller sample give a larger (less useful) credible set?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

def compute_credible_set(phenotype, genotypes, prior_variance=0.04):
    betas, std_errors, _ = run_gwas(phenotype, genotypes)
    z = betas / std_errors
    sampling_variance = std_errors**2
    r = prior_variance / (sampling_variance + prior_variance)
    log_abf = 0.5*np.log(1 - r) + 0.5*z**2*r
    pip = np.exp(log_abf - log_abf.max()); pip /= pip.sum()
    order = np.argsort(pip)[::-1]
    cumulative_pip = np.cumsum(pip[order])
    return order[:np.searchsorted(cumulative_pip, 0.95) + 1]

rng = np.random.default_rng(0)
for fraction in [1.0, 0.5, 0.25]:
    n = int(fraction * len(yA))
    subsample = rng.choice(len(yA), n, replace=False)
    cs = compute_credible_set(yA[subsample], GA[subsample])
    print(f"N={n:5d}: credible-set size = {len(cs):3d}, contains causal = {causalA in cs.tolist()}")


<details>
<summary>👉 <b>Show answer</b></summary>

A smaller sample gives **noisier effect estimates**, so the data can't distinguish the causal variant from its LD neighbours as sharply — the posterior probability spreads across more variants, inflating the credible set. More samples sharpen the signal and shrink the set.

</details>

### Challenge 3: When one causal variant isn't enough — \~12 min

The single-causal credible set assumes exactly **one** causal variant per locus. **Locus B** has
two. Build the single-causal credible set for Locus B and check whether it captures **both**
causals — then use conditional analysis to recover the second signal (the idea behind SuSiE).


In [ ]:
# Challenge: two causal variants (Locus B).
# Step 1. Build the single-causal credible set, exactly as in Part 3.
betas, std_errors, pvalues = run_gwas(yB, GB)
z = betas / std_errors
prior_variance = 0.04
sampling_variance = std_errors**2
r = prior_variance / (sampling_variance + prior_variance)
log_abf = 0.5*np.log(1 - r) + 0.5*z**2*r
abf = np.exp(log_abf - log_abf.max())
pip = abf / abf.sum()

order = np.argsort(pip)[::-1]
cumulative_pip = np.cumsum(pip[order])
credible_set = order[:np.searchsorted(cumulative_pip, 0.95) + 1]

print(f"True causals: {causalB.tolist()}")
print(f"Single-causal 95% credible set: {sorted(credible_set.tolist())}")
print(f"  captures BOTH causals? {all(c in credible_set.tolist() for c in causalB)}")

# Step 2. Condition on the lead variant, then refit — does the SECOND causal now stand out?
lead_variant   = int(np.argmin(pvalues))
lead_genotype  = GB[:, [lead_variant]]          # genotype of the lead, as a single covariate column
# YOUR CODE HERE: re-run the GWAS, passing the lead genotype as a covariate.
_, _, pvalues_conditional = run_gwas(yB, GB, covars=???)

new_lead = int(np.argmin(pvalues_conditional))
print(f"After conditioning on variant {lead_variant}: new lead = {new_lead} "
      f"(near the other causal {causalB.tolist()})")
print("Q: Why does a single-causal credible set fail when there are two signals?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

betas, std_errors, pvalues = run_gwas(yB, GB)
z = betas / std_errors
prior_variance = 0.04
sampling_variance = std_errors**2
r = prior_variance / (sampling_variance + prior_variance)
log_abf = 0.5*np.log(1 - r) + 0.5*z**2*r
abf = np.exp(log_abf - log_abf.max())
pip = abf / abf.sum()

order = np.argsort(pip)[::-1]
cumulative_pip = np.cumsum(pip[order])
credible_set = order[:np.searchsorted(cumulative_pip, 0.95) + 1]

print(f"True causals: {causalB.tolist()}")
print(f"Single-causal 95% credible set: {sorted(credible_set.tolist())}")
print(f"  captures BOTH causals? {all(c in credible_set.tolist() for c in causalB)}")

lead_variant   = int(np.argmin(pvalues))
lead_genotype  = GB[:, [lead_variant]]
_, _, pvalues_conditional = run_gwas(yB, GB, covars=lead_genotype)

new_lead = int(np.argmin(pvalues_conditional))
print(f"After conditioning on variant {lead_variant}: new lead = {new_lead} "
      f"(near the other causal {causalB.tolist()})")
# A single-causal model puts all PIP near one signal; SuSiE fits several single-effect components
# iteratively (much like conditioning) to give one credible set per causal variant.


<details>
<summary>👉 <b>Show answer</b></summary>

A single-causal-variant assumption can't represent **two** independent signals: it tries to explain both with one variant, often landing on a compromise variant that is causal for neither, or missing one signal entirely. You need a multi-causal method (e.g. SuSiE / FINEMAP with several allowed signals) to recover both credible sets.

</details>